In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import gc
import os
import sys

In [3]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("env ready GPU-Cache is cleared")

env ready GPU-Cache is cleared


In [4]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
output_path = "/content/drive/MyDrive/llm_from_scratch/text_generation_model/title_model"

sys.path.append(proj_path)
os.chdir(proj_path)
print(f"cwd switched to: {os.getcwd()}")


cwd switched to: /content/drive/MyDrive/llm_from_scratch/src


In [5]:
import torch
import tiktoken
from torch.utils.data import Dataset
from datasets import load_dataset

class DatasetProcessing(Dataset):
    def __init__(self, raw_dataset, max_length=512):
        # using our GPT-2 tokenizer matched to our custom architecture.
        self.encoding = tiktoken.get_encoding('gpt2')
        self.max_length = max_length
        self.raw_dataset = raw_dataset

    def __len__(self):
        return len(self.raw_dataset)

    def __getitem__(self, idx):
        item = self.raw_dataset[idx]
        conversations_list = item['conversations']

        #rconstructing the user and chatbot multi-turn conversation
        chat_history = ""
        user_demands = []

        for turn in conversations_list:
            speaker_role = turn['from']
            text_content = turn['value'].strip()

            if speaker_role == "system":
                continue # we keep input clean here if it is from system
            elif speaker_role == "human":
                chat_history += f"user: {text_content}\n"
                user_demands.append(text_content)
            elif speaker_role == "gpt":
                chat_history += f"assistant: {text_content}\n"

        raw_target_text = conversations_list[-1]['value'].strip()
        words = [w for w in raw_target_text.replace("\n", " ").split(" ") if w]

        # a meaningful slice of the output for the target title
        if len(words) > 7:
            title_target = " ".join(words[:5]).strip(".,!? ")
        else:
            title_target = " ".join(words).strip(".,!? ")

        user_demands_str = " ".join(user_demands).lower()
        if any(keyword in user_demands_str for keyword in ["how", "write", "code", "delete", "run", "fix"]):
            instruction = "Read the following chat log and provide a short title describing what the user wants or what is being discussed."
        else:
            instruction = "Read the following conversation and generate a short declarative title summarizing its core conceptual theme."

        # template structure split into prompt and target
        prompt_text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Conversation:\n{chat_history.strip()}\n\n"
            f"### Title:\n"
        )

        # encoding them separately to find the exact boundary
        prompt_tokens = self.encoding.encode(prompt_text)
        target_tokens = self.encoding.encode(title_target)

        if len(prompt_tokens) + len(target_tokens) > self.max_length:
            # calculating how many tokens we can allow the prompt to have
            available_prompt_slots = self.max_length - len(target_tokens)

            # truncating the last part of the conversation, keeping the vital target title intact
            prompt_tokens = prompt_tokens[:available_prompt_slots]

        #combine for input
        input_ids = prompt_tokens + target_tokens

        #masking the lables with -100 to remove them from being part in loss calcualtion
        labels = [-100] * len(prompt_tokens) + target_tokens


        return {
            'input_ids': input_ids,
            'labels': labels
        }

print("loading dataset from Hugging Face...")
shared_download = load_dataset("Open-Orca/SlimOrca", split="train")

train_slice = shared_download.select(range(0, 12000))
val_slice = shared_download.select(range(12000, 14000))

# creating raw dataset into preprocessed ready for training dataset:
train_dataset = DatasetProcessing(train_slice, max_length=512)
val_dataset = DatasetProcessing(val_slice, max_length=512)

print("data has been loaded successfully.")

loading dataset from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

oo-labeled_correct.gpt4.sharegpt.jsonl:   0%|          | 0.00/986M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/517982 [00:00<?, ? examples/s]

data has been loaded successfully.


In [8]:
print(train_dataset[44])

{'input_ids': [21017, 46486, 25, 198, 5569, 262, 1708, 8537, 2604, 290, 2148, 257, 1790, 3670, 12059, 644, 262, 2836, 3382, 393, 644, 318, 852, 6693, 13, 198, 198, 21017, 42427, 25, 198, 7220, 25, 4222, 3280, 262, 1708, 1808, 25, 10854, 25, 220, 532, 5524, 9791, 4245, 12584, 18017, 357, 9437, 11, 3623, 11, 3503, 2014, 532, 23699, 17556, 422, 262, 9482, 12584, 18017, 3802, 262, 8137, 532, 383, 787, 12, 929, 286, 262, 8137, 2458, 532, 2773, 2568, 422, 262, 4252, 2314, 6654, 736, 832, 262, 3421, 8137, 532, 383, 4534, 6140, 284, 4894, 780, 286, 262, 3131, 13640, 2568, 532, 5524, 9791, 2005, 866, 7150, 532, 5524, 9791, 466, 407, 302, 12, 15060, 262, 7150, 532, 34925, 2314, 1037, 1011, 262, 3131, 6588, 17556, 422, 262, 8137, 532, 383, 3131, 13640, 2568, 4940, 284, 1487, 262, 4534, 5, 2, 87, 1983, 26, 82, 1790, 3381, 6193, 532, 16178, 262, 890, 12, 4354, 4258, 7572, 923, 284, 1487, 532, 220, 220, 350, 861, 5945, 341, 14078, 25, 11691, 517, 14903, 8833, 4325, 11, 703, 481, 340, 2689, 517, 4815

In [9]:
import torch
from transformers.generation.utils import GenerationMixin
from gpt_model import GPTModel
from config import GPTConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#dynamically patching GPTModel to inherit GenerationMixin
if GenerationMixin not in GPTModel.__bases__:
    print("injecting GenerationMixin into GPTModel base structural layers...")
    GPTModel.__bases__ = (GPTModel.__bases__[0], GenerationMixin) + GPTModel.__bases__[1:]

CONFIG = GPTConfig()

# initialising model
model = GPTModel(CONFIG)

# loading weights from pretrained weights form .pth file
CORRECT_PRETRAINED_PATH = "/content/drive/MyDrive/llm_from_scratch/pretrained_weights.pth"
model.load_state_dict(torch.load(CORRECT_PRETRAINED_PATH, map_location=device))

print("model is initialized and weights are successfully loaded.")

injecting GenerationMixin into GPTModel base structural layers...
model is initialized and weights are successfully loaded.


In [11]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [12]:
from peft import LoraConfig, get_peft_model

def setup_lora_model(model):
    target_modules = ["W_query", "W_key", "W_value", "out_proj", "out_head"]

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        modules_to_save=None
    )
    return get_peft_model(model, lora_config)

model = setup_lora_model(model)
model.to(device)
print("LoRA configurations attached to weight channels.")


LoRA configurations attached to weight channels.


In [13]:
class DynamicDataCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        #to find maximum sequence length present in each specific batch
        batch_lens = [len(f['input_ids']) for f in features]
        max_batch_len = max(batch_lens)

        batch_input_ids = []
        batch_labels = []

        for f in features:
            inputs = f['input_ids']
            labels = f['labels']
            remainder = max_batch_len - len(inputs)

            # pading inputs with eot token and labels with -100
            padded_inputs = inputs + [self.pad_token_id] * remainder
            padded_labels = labels + [-100] * remainder

            batch_input_ids.append(padded_inputs)
            batch_labels.append(padded_labels)

        return {
            'input_ids': torch.tensor(batch_input_ids, dtype=torch.long),
            'labels': torch.tensor(batch_labels, dtype=torch.long)
        }

# initializing the collator using tiktoken eot value
data_collator = DynamicDataCollator(pad_token_id=train_dataset.encoding.eot_token)


In [ ]:
from transformers import TrainingArguments, Trainer

os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir=output_path,
    num_train_epochs=2,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    fp16=True,
    logging_steps=10,
    remove_unused_columns=False,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator # Tells Trainer to use dynamic padding per batch
)


print("training statring ...")
trainer.train()
print("training completed.")

training statring ...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


In [ ]:
# merging LoRA weights with existing ones
merged_model = model.merge_and_unload()

#saving treined weights
final_save_path = "/content/drive/MyDrive/llm_from_scratch/text_generation_model/title_model/chat_title_generator_model_final.pth"
torch.save(merged_model.state_dict(), final_save_path)

print(f"successfully weights file generated at: {final_save_path}")
